In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Loading and Preparation

The dataset consisted of song lyrics from 21 artists, each stored in a separate CSV file.
The first step was loading and exploring a single file to understand the structure before combining them all.


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/ArianaGrande.csv')

print(df.head())

print(df.shape)
print(df.isnull().sum())

print(df['Lyric'][0])

          Artist                   Title            Album        Date  \
0  Ariana Grande          ​thank u, next    thank u, next  2018-11-03   
1  Ariana Grande                 7 rings    thank u, next  2019-01-18   
2  Ariana Grande         ​God is a woman        Sweetener  2018-07-13   
3  Ariana Grande            Side To Side  Dangerous Woman  2016-05-20   
4  Ariana Grande  ​​no tears left to cry        Sweetener  2018-04-20   

                                               Lyric    Year  
0  thought i'd end up with sean but he wasn't a m...  2018.0  
1  yeah breakfast at tiffany's and bottles of bub...  2019.0  
2  you you love it how i move you you love it how...  2018.0  
3  ariana grande  nicki minaj i've been here all ...  2016.0  
4  right now i'm in a state of mind i wanna be in...  2018.0  
(308, 6)
Artist     0
Title      0
Album     93
Date      63
Lyric      0
Year      63
dtype: int64
thought i'd end up with sean but he wasn't a match wrote some songs about ricky now

## Combining Artist Files

Since each artist had their own CSV file, glob was used to find all 21 files
and load them into a single combined dataframe.

In [ ]:
import glob

files = glob.glob('/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/*.csv')
print(files)

list_of_dfs = []

for file in files:
    df = pd.read_csv(file)
    list_of_dfs.append(df)

combined_df = pd.concat(list_of_dfs, ignore_index=True)


print(combined_df.shape)
print(combined_df['Artist'].value_counts())


['/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/BTS.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/Rihanna.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/ColdPlay.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/NickiMinaj.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/DuaLipa.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/PostMalone.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/SelenaGomez.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/CardiB.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/Drake.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/Khalid.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/Beyonce.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/LadyGaga.csv', '/content/drive/MyDrive/NaturalLanguageProcessing/Data/csv/Maroon5.csv', '/content/drive/MyDrive/NaturalLanguageProcess

## Data Cleaning

Selected only the relevant columns, removed songs with missing lyrics,
filtered out songs with fewer than 60 words, and truncated lyrics to 180 words
to stay within the model's token limit.

In [ ]:
combined_df = combined_df[['Artist', 'Title', 'Lyric']]
combined_df = combined_df.dropna(subset=['Lyric'])
combined_df = combined_df[combined_df['Lyric'].apply(lambda x: len(x.split()) >= 60)]
combined_df['Lyric'] = combined_df['Lyric'].apply(lambda x: ' '.join(x.split()[:180]))

print(combined_df.shape)
print(combined_df.isnull().sum())

(5442, 3)
Artist    0
Title     0
Lyric     0
dtype: int64


remvoing any null values



In [ ]:
combined_df['Lyric'] = combined_df['Lyric'].apply(lambda x: ' '.join(x.split()[:180]))
print(combined_df['Lyric'][0][:180])

jungkook 'cause i i i'm in the stars tonight so watch me bring the fire and set the night alight jungkook shoes on get up in the morn' cup of milk let's rock and roll king kong kic


Loading the Model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Lyrics_Descriptions = combined_df['Lyric'].tolist()

doc_embeddings = st_model.encode(Lyrics_Descriptions, convert_to_tensor=True)


query = "beautiful day, love, sunshine"
query_embedding = st_model.encode(query, convert_to_tensor=True)


print(query_embedding)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tensor([-6.6573e-02,  7.9952e-02,  1.1695e-01,  4.6646e-03,  1.6635e-02,
         5.9108e-02,  1.0833e-01, -7.7220e-02,  4.4372e-02, -4.0480e-02,
         6.0811e-03,  1.1253e-02,  1.9025e-02,  9.8935e-03,  5.6469e-02,
         6.9041e-02, -2.2121e-02, -1.1259e-02, -1.5427e-02,  8.1503e-03,
        -6.3825e-02,  1.6188e-02, -5.3253e-02,  2.7313e-02, -6.6364e-02,
         1.3482e-01,  2.6260e-02,  3.6730e-02, -6.2593e-02, -3.4265e-02,
         2.7095e-02,  5.8049e-02,  3.6566e-02,  1.8916e-03, -3.7676e-02,
         1.3441e-02, -2.2418e-02, -1.2095e-01, -3.0338e-02,  3.5948e-02,
        -5.8482e-02, -9.5263e-02,  5.4675e-03,  2.4257e-02, -4.3974e-02,
        -2.6479e-02,  2.0634e-02, -4.7058e-02,  1.1730e-01,  4.6684e-02,
        -5.9013e-02, -5.7515e-03, -9.9608e-02, -1.4594e-03,  1.0945e-02,
         7.1400e-02, -1.7160e-02, -1.8652e-02,  1.1861e-01,  2.7549e-02,
         1.9665e-02,  4.3615e-02,  2.4223e-02,  5.1164e-02,  7.4221e-02,
        -2.1598e-02, -5.2656e-02, -3.6644e-02, -6.5

In [ ]:
def cosine(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

print("Query:", query)

for d_idx in range(len(Lyrics_Descriptions)):
    doc_sim_score = cosine(query_embedding, doc_embeddings[d_idx])
    song = combined_df.iloc[d_idx]
    print(f"{doc_sim_score.item():.3f} | {song['Artist']} - {song['Title']}")
    print(f"{song['Lyric'][:150]}")
    print("---")

Streaming output truncated to the last 5000 lines.
yeah yeah yeah happy birthday that hold on tight yeah that hold on tight ooh that hold on tight them lips won't let me go lips won't let me go lips wo
---
0.177 | Justin Bieber - Where Are Ü Now - Piano Version
justin bieber i need you the m i need you i need you the m i need you i need the m the m the m the m the m the m i need you the m i need you i need yo
---
0.211 | Justin Bieber - Trust In Me
i don't wanna go to sleep i don't wanna go to sleep 'cause then the day would end then the day would end i want this to last to last forever and the c
---
0.220 | Justin Bieber - Life Is Worth Living (Acoustic)
ended up on a crossroad tried to figure out which way to go it's like you're stuck on a treadmill running in the same place you got your hazard lights
---
0.263 | Justin Bieber - One Love - Single Version
ohayeohaye ohayeohaye ohayeohaye yeah yeah yeah yeah i won't let the night stand in my way i know what i want i know what i can yea

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Lyrics_Descriptions = combined_df['Lyric'].tolist()

doc_embeddings = st_model.encode(Lyrics_Descriptions, convert_to_tensor=True)


query = "heartbroken, glum, sad"
query_embedding = st_model.encode(query, convert_to_tensor=True)


print(query_embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([ 2.6572e-02, -1.7458e-02,  6.7334e-02,  8.4217e-03,  3.9765e-02,
        -7.3275e-02,  5.8666e-02,  1.8372e-02,  8.5787e-02, -7.0624e-02,
         1.3659e-02, -4.3373e-02, -5.8119e-02, -2.8635e-02,  7.1873e-02,
        -7.4678e-02,  5.9433e-03, -1.1261e-01, -1.3195e-01,  3.9462e-02,
        -6.9908e-02,  3.2321e-02, -9.2931e-02,  2.2098e-02,  5.1032e-03,
         3.9365e-02,  1.9993e-02, -2.2608e-03, -5.9273e-02,  4.4154e-02,
         1.4831e-02,  3.9593e-02,  8.3921e-02,  6.6961e-02,  4.0737e-02,
         4.6145e-02, -1.0611e-01,  5.8351e-02, -1.9972e-02, -5.1532e-02,
        -3.1624e-02, -5.5669e-02,  5.6638e-02,  5.6855e-03, -1.7591e-02,
        -6.3541e-02, -7.5277e-03, -4.4694e-02, -1.1880e-03,  1.8250e-02,
        -3.6753e-02, -3.2129e-02, -1.1756e-01,  2.9262e-02,  7.1524e-02,
         3.4371e-02, -1.0563e-03,  5.5837e-02,  4.3417e-02, -2.4538e-02,
         4.4661e-02,  1.3307e-02, -4.6198e-02,  3.7275e-02,  2.9861e-02,
         2.8423e-02,  2.6751e-02, -5.5457e-02, -8.2

In [ ]:
def cosine(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

print("Query:", query)

for d_idx in range(len(Lyrics_Descriptions)):
    doc_sim_score = cosine(query_embedding, doc_embeddings[d_idx])
    song = combined_df.iloc[d_idx]
    print(f"{doc_sim_score.item():.3f} | {song['Artist']} - {song['Title']}")
    print(f"{song['Lyric'][:150]}")
    print("---")

Streaming output truncated to the last 5000 lines.
yeah yeah yeah happy birthday that hold on tight yeah that hold on tight ooh that hold on tight them lips won't let me go lips won't let me go lips wo
---
0.227 | Justin Bieber - Where Are Ü Now - Piano Version
justin bieber i need you the m i need you i need you the m i need you i need the m the m the m the m the m the m i need you the m i need you i need yo
---
0.212 | Justin Bieber - Trust In Me
i don't wanna go to sleep i don't wanna go to sleep 'cause then the day would end then the day would end i want this to last to last forever and the c
---
0.364 | Justin Bieber - Life Is Worth Living (Acoustic)
ended up on a crossroad tried to figure out which way to go it's like you're stuck on a treadmill running in the same place you got your hazard lights
---
0.179 | Justin Bieber - One Love - Single Version
ohayeohaye ohayeohaye ohayeohaye yeah yeah yeah yeah i won't let the night stand in my way i know what i want i know what i can yea

## Improved Approach

The initial approach printed results in dataset order with no ranking,
making it impossible to find the best matches. Using `torch.topk`,
the results are now ranked by cosine similarity score, returning only
the top 10 most relevant songs for any mood query.

In [ ]:
from sentence_transformers import util
util.cos_sim(query_embedding, doc_embeddings)

tensor([[0.1488, 0.2300, 0.0958,  ..., 0.1413, 0.0529, 0.2870]],
       device='cuda:0')

In [ ]:
query = "heartbroken, glum, sad"
query_embedding = st_model.encode(query, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, doc_embeddings).squeeze()
top_results = torch.topk(scores, k=10)

print("Query:", query)
for idx in top_results.indices:
    song = combined_df.iloc[idx.item()]
    print(f"{song['Artist']} - {song['Title']}")
    print(f"{song['Lyric'][:150]}")
    print("---")

Query: heartbroken, glum, sad
Lady Gaga - Smile (One World: Together At Home)
smile though your heart is breaking smile even though it's aching when there are clouds in the sky you'll get by if you smile through your fear and so
---
Justin Bieber - Heartache
what wait but yesterday we were i don't understand we started out it was perfect nothing but fun and my heart was convinced to say that you're the one
---
Taylor Swift - Why She Disappeared [Poem]
when she fell she fell apart cracked her bones on the pavement she once decorated as a child with sidewalk chalk when she crashed her clothes disinteg
---
Khalid - ​wildflower (rough)
mm yeah mm yeah mm yeah yeah spend my days countin' down time will pass that i know whatcha done broke me down not alone im vulnerable let you in take
---
Maroon 5 - This Love
i was so high i did not recognize the fire burning in her eyes the chaos that controlled my mind whispered goodbye as she got on a plane never to retu
---
Taylor Swift - Sad Beautiful 

In [ ]:
query = "Happy, Joyful, elated"
query_embedding = st_model.encode(query, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, doc_embeddings).squeeze()
top_results = torch.topk(scores, k=10)

print("Query:", query)
for idx in top_results.indices:
    song = combined_df.iloc[idx.item()]
    print(f"{song['Artist']} - {song['Title']}")
    print(f"{song['Lyric'][:150]}")
    print("---")

Query: Happy, Joyful, elated
Ariana Grande - ​pete davidson
mmm yeah yuh i thought you into my life whoa look at my mind yuh no better place or a time look how they align unimust have my back fell from the sky 
---
Rihanna - Happy
just as long as it makes you happy if it makes you happy just be happy just as long as it makes you happy if it makes you happy just be happy it's lik
---
Coldplay - Colour Spectrum (Live In Buenos Aires)
spoken this being human is a guest house every morning a new arrival a joy a depression a meanness some momentary awareness comes as an unexpected vis
---
Maroon 5 - Happy (BBC Radio 1 Live Lounge)
it might seem crazy what i'm 'bout to say sunshine she's here you can take a break yeah hot air balloon that could go to space yeah with the air like 
---
Justin Bieber - Purpose
feeling like i'm breathing my last breath feeling like i'm walking my last steps look at all of these tears i've wept look at all the promises that i'
---
Beyoncé - God Made You Beautiful